# Solar Power Generation Prediction Using Machine Learning with Weather-Based Feature Analysis

**Project type:** Supervised regression  
**Dataset:** Indian Solar Power Generation Data — Plant 1  
**Target:** Total plant-level AC power output (`AC_POWER`, kW)  
**Prediction objective:** Estimate AC power at the same timestamp using measured weather and time-based features.

## Automatic setup
This notebook downloads the public Kaggle dataset automatically using `kagglehub`. You do **not** need to download or extract the dataset manually. An internet connection is required in the notebook environment.

## KDD workflow
1. Data selection and understanding
2. Data preprocessing
3. Data transformation and feature engineering
4. Predictive modelling
5. Evaluation
6. Interpretation of results

This is same-timestamp estimation using measured weather, not future forecasting. The notebook excludes other generation readings such as `DC_POWER`, `DAILY_YIELD`, and `TOTAL_YIELD` from the model inputs to reduce leakage.

## 1. Install and import libraries

In [ ]:
%pip -q install kagglehub

import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

## 2. Automatically download the dataset

In [ ]:
# Downloads the public Kaggle dataset into the notebook environment's cache.
# No manual ZIP download or extraction is required.
dataset_path = Path(kagglehub.dataset_download("anikannal/solar-power-generation-data"))
print("Dataset downloaded to:", dataset_path)

# Search recursively in case Kaggle stores CSV files in a nested folder.
generation_matches = list(dataset_path.rglob("Plant_1_Generation_Data.csv"))
weather_matches = list(dataset_path.rglob("Plant_1_Weather_Sensor_Data.csv"))

if not generation_matches or not weather_matches:
    available_csvs = [str(p) for p in dataset_path.rglob("*.csv")]
    raise FileNotFoundError(
        "Expected Plant 1 CSV files were not found. CSV files discovered:\n"
        + "\n".join(available_csvs[:30])
    )

GENERATION_FILE = generation_matches[0]
WEATHER_FILE = weather_matches[0]

print("Generation file:", GENERATION_FILE)
print("Weather file:", WEATHER_FILE)

generation = pd.read_csv(GENERATION_FILE)
weather = pd.read_csv(WEATHER_FILE)

print("Generation shape:", generation.shape)
print("Weather shape:", weather.shape)
display(generation.head())
display(weather.head())

## 3. Understand and validate the data

In [ ]:
print("Generation columns:")
print(generation.columns.tolist())

print("\nWeather columns:")
print(weather.columns.tolist())

print("\nGeneration data types:")
display(generation.dtypes.to_frame("data_type"))

print("\nWeather data types:")
display(weather.dtypes.to_frame("data_type"))

print("\nMissing values — generation:")
display(generation.isna().sum().to_frame("missing_count"))

print("\nMissing values — weather:")
display(weather.isna().sum().to_frame("missing_count"))

print("\nDuplicate rows — generation:", generation.duplicated().sum())
print("Duplicate rows — weather:", weather.duplicated().sum())

## 4. Parse timestamps and validate required columns

In [ ]:
required_generation = {"DATE_TIME", "PLANT_ID", "SOURCE_KEY", "AC_POWER"}
required_weather = {
    "DATE_TIME", "PLANT_ID", "AMBIENT_TEMPERATURE",
    "MODULE_TEMPERATURE", "IRRADIATION"
}

missing_gen_cols = required_generation - set(generation.columns)
missing_weather_cols = required_weather - set(weather.columns)

if missing_gen_cols:
    raise ValueError(f"Missing expected generation columns: {missing_gen_cols}")
if missing_weather_cols:
    raise ValueError(f"Missing expected weather columns: {missing_weather_cols}")

generation = generation.drop_duplicates().copy()
weather = weather.drop_duplicates().copy()

generation["DATE_TIME"] = pd.to_datetime(generation["DATE_TIME"], errors="coerce")
weather["DATE_TIME"] = pd.to_datetime(weather["DATE_TIME"], errors="coerce")

generation["AC_POWER"] = pd.to_numeric(generation["AC_POWER"], errors="coerce")
for col in ["AMBIENT_TEMPERATURE", "MODULE_TEMPERATURE", "IRRADIATION"]:
    weather[col] = pd.to_numeric(weather[col], errors="coerce")

generation = generation.dropna(subset=["DATE_TIME", "PLANT_ID", "AC_POWER"])
weather = weather.dropna(subset=["DATE_TIME", "PLANT_ID"])

print("Generation timestamp range:",
      generation["DATE_TIME"].min(), "to", generation["DATE_TIME"].max())
print("Weather timestamp range:",
      weather["DATE_TIME"].min(), "to", weather["DATE_TIME"].max())
print("Plant IDs in generation data:", generation["PLANT_ID"].unique())
print("Plant IDs in weather data:", weather["PLANT_ID"].unique())

## 5. Aggregate inverter readings to plant-level output

In [ ]:
# Plant 1 only. Sum inverter AC power at each timestamp.
plant_generation = (
    generation
    .groupby(["PLANT_ID", "DATE_TIME"], as_index=False)
    .agg(AC_POWER=("AC_POWER", "sum"))
)

# Average weather readings if multiple sensors report at the same timestamp.
weather_features = [
    "AMBIENT_TEMPERATURE",
    "MODULE_TEMPERATURE",
    "IRRADIATION"
]
plant_weather = (
    weather
    .groupby(["PLANT_ID", "DATE_TIME"], as_index=False)[weather_features]
    .mean()
)

print("Plant-level generation shape:", plant_generation.shape)
print("Timestamp-level weather shape:", plant_weather.shape)
display(plant_generation.head())
display(plant_weather.head())

## 6. Merge weather and generation

In [ ]:
data = pd.merge(
    plant_generation,
    plant_weather,
    on=["PLANT_ID", "DATE_TIME"],
    how="inner",
    validate="one_to_one"
).sort_values("DATE_TIME").reset_index(drop=True)

if data.empty:
    raise ValueError(
        "No matching plant/timestamp rows after merging. Check the downloaded files."
    )

print("Merged dataset shape:", data.shape)
print("Date range:", data["DATE_TIME"].min(), "to", data["DATE_TIME"].max())
display(data.head())
display(data.isna().sum().to_frame("missing_count"))

## 7. Exploratory data analysis

In [ ]:
display(data[["AC_POWER"] + weather_features].describe().T)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes[0, 0].hist(data["AC_POWER"].dropna(), bins=40)
axes[0, 0].set_title("Distribution of Plant AC Power")
axes[0, 0].set_xlabel("AC Power (kW)")

axes[0, 1].scatter(data["IRRADIATION"], data["AC_POWER"], alpha=0.35)
axes[0, 1].set_title("Irradiation vs AC Power")
axes[0, 1].set_xlabel("Irradiation")
axes[0, 1].set_ylabel("AC Power (kW)")

axes[1, 0].scatter(data["AMBIENT_TEMPERATURE"], data["AC_POWER"], alpha=0.35)
axes[1, 0].set_title("Ambient Temperature vs AC Power")
axes[1, 0].set_xlabel("Ambient temperature")
axes[1, 0].set_ylabel("AC Power (kW)")

axes[1, 1].scatter(data["MODULE_TEMPERATURE"], data["AC_POWER"], alpha=0.35)
axes[1, 1].set_title("Module Temperature vs AC Power")
axes[1, 1].set_xlabel("Module temperature")
axes[1, 1].set_ylabel("AC Power (kW)")

plt.tight_layout()
plt.show()

## 8. Feature engineering and leakage prevention

In [ ]:
data["HOUR"] = data["DATE_TIME"].dt.hour
data["MINUTE"] = data["DATE_TIME"].dt.minute
data["DAY_OF_WEEK"] = data["DATE_TIME"].dt.dayofweek

FEATURES = [
    "IRRADIATION",
    "AMBIENT_TEMPERATURE",
    "MODULE_TEMPERATURE",
    "HOUR",
    "MINUTE",
    "DAY_OF_WEEK"
]
TARGET = "AC_POWER"

model_data = data.dropna(subset=[TARGET]).copy().reset_index(drop=True)
X = model_data[FEATURES]
y = model_data[TARGET]

print("Input features:", FEATURES)
print("Target:", TARGET)
print("Usable rows:", len(model_data))
print("Excluded from inputs: DC_POWER, DAILY_YIELD, TOTAL_YIELD, SOURCE_KEY")

## 9. Chronological train/test split

In [ ]:
split_index = int(len(model_data) * 0.80)
if split_index <= 0 or split_index >= len(model_data):
    raise ValueError("Not enough observations for an 80/20 chronological split.")

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training period:", model_data["DATE_TIME"].iloc[0],
      "to", model_data["DATE_TIME"].iloc[split_index - 1])
print("Testing period:", model_data["DATE_TIME"].iloc[split_index],
      "to", model_data["DATE_TIME"].iloc[-1])

## 10. Train three regression models

In [ ]:
models = {
    "Linear Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=200, min_samples_leaf=2,
            random_state=RANDOM_STATE, n_jobs=-1
        ))
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingRegressor(
            n_estimators=150, learning_rate=0.05,
            max_depth=3, random_state=RANDOM_STATE
        ))
    ])
}

predictions = {}
results = []

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    predictions[name] = pred
    results.append({
        "Model": name,
        "MAE (kW)": mean_absolute_error(y_test, pred),
        "RMSE (kW)": np.sqrt(mean_squared_error(y_test, pred)),
        "R²": r2_score(y_test, pred)
    })

results_df = pd.DataFrame(results).sort_values("RMSE (kW)").reset_index(drop=True)
display(results_df.round(4))

## 11. Compare models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(results_df["Model"], results_df["MAE (kW)"])
axes[0].set_title("Model Comparison — MAE")
axes[0].set_ylabel("MAE (kW)")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(results_df["Model"], results_df["RMSE (kW)"])
axes[1].set_title("Model Comparison — RMSE")
axes[1].set_ylabel("RMSE (kW)")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
best_predictions = predictions[best_model_name]
print("Best model by test RMSE:", best_model_name)

## 12. Actual vs predicted output

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_predictions, alpha=0.5)
lower = min(float(y_test.min()), float(np.min(best_predictions)))
upper = max(float(y_test.max()), float(np.max(best_predictions)))
plt.plot([lower, upper], [lower, upper], linestyle="--")
plt.xlabel("Actual AC Power (kW)")
plt.ylabel("Predicted AC Power (kW)")
plt.title(f"Actual vs Predicted — {best_model_name}")
plt.tight_layout()
plt.show()

## 13. Residual analysis

In [ ]:
residuals = y_test.to_numpy() - best_predictions
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].scatter(best_predictions, residuals, alpha=0.5)
axes[0].axhline(0, linestyle="--")
axes[0].set_xlabel("Predicted AC Power (kW)")
axes[0].set_ylabel("Residual (actual - predicted)")
axes[0].set_title("Residuals vs Predicted")

axes[1].hist(residuals, bins=35)
axes[1].set_xlabel("Residual (kW)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Residual Distribution")

plt.tight_layout()
plt.show()

## 14. Feature importance

In [ ]:
tree_candidates = ["Random Forest", "Gradient Boosting"]
best_tree_name = min(
    tree_candidates,
    key=lambda name: float(
        results_df.loc[results_df["Model"] == name, "RMSE (kW)"].iloc[0]
    )
)
tree_model = models[best_tree_name].named_steps["model"]

importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": tree_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance)

plt.figure(figsize=(8, 4.5))
plt.barh(importance["Feature"][::-1], importance["Importance"][::-1])
plt.xlabel("Feature importance")
plt.title(f"Feature Importance — {best_tree_name}")
plt.tight_layout()
plt.show()

print("Feature importance indicates predictive usefulness, not causation.")

## 15. Try an illustrative prediction

In [ ]:
sample_observation = pd.DataFrame([{
    "IRRADIATION": 0.60,
    "AMBIENT_TEMPERATURE": 28.0,
    "MODULE_TEMPERATURE": 40.0,
    "HOUR": 12,
    "MINUTE": 0,
    "DAY_OF_WEEK": 2
}])

sample_prediction = best_model.predict(sample_observation[FEATURES])[0]
print(f"Illustrative predicted plant AC power: {sample_prediction:.2f} kW")
print("These are example inputs, not verified measurements from the dataset.")

## 16. Results summary

In [ ]:
best_row = results_df.iloc[0]
print("PROJECT RESULTS SUMMARY")
print("-----------------------")
print(f"Best model by RMSE: {best_row['Model']}")
print(f"Test MAE: {best_row['MAE (kW)']:.3f} kW")
print(f"Test RMSE: {best_row['RMSE (kW)']:.3f} kW")
print(f"Test R²: {best_row['R²']:.4f}")
print("\nInterpretation:")
print("- Lower MAE and RMSE mean smaller prediction errors.")
print("- R² closer to 1 indicates better fit on this test set.")
print("- Negative R² means worse performance than predicting the test-set mean.")

## 17. Conclusion — complete after running
Use the actual outputs to report the best model, MAE, RMSE, R², important features, and patterns in the prediction errors. Do not invent performance values.

### Limitations
- Only one plant is used.
- The data covers a short period, so results do not establish year-round performance.
- This is same-timestamp estimation using measured weather, not a future forecast.
- A chronological test split better reflects later-time evaluation than a random split.